In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
print(torch.cuda.get_device_name(0))


In [ ]:
import os
import wget
import zipfile

os.makedirs("coco", exist_ok=True)

url = "http://images.cocodataset.org/zips/val2014.zip"
zip_path = "coco/val2014.zip"

# Download
wget.download(url, zip_path)

# Extract
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("coco/")

In [ ]:
import os

count = len(os.listdir("coco/val2014"))
print(count)

In [ ]:
from PIL import Image
import random
import os

files = os.listdir("coco/val2014")
img_path = "coco/val2014/" + random.choice(files)

img = Image.open(img_path)
img.show()

print("Opened:", img_path)

In [ ]:
print(files[0])

In [ ]:
import json

with open("annotations/instances_val2014.json") as f:
    coco = json.load(f)

print(len(coco["images"]))
print(len(coco["annotations"]))
print(len(coco["categories"]))

In [ ]:
import json
import random
from collections import defaultdict

# load annotations
with open("annotations/instances_val2014.json") as f:
    coco = json.load(f)

# build image → objects mapping
img_to_objs = defaultdict(set)

for ann in coco["annotations"]:
    img_to_objs[ann["image_id"]].add(ann["category_id"])

# category id → name
cat_map = {c["id"]: c["name"] for c in coco["categories"]}
all_categories = list(cat_map.keys())

data = []

# 🔥 MAIN LOOP
for img in coco["images"]:
    img_id = img["id"]
    filename = img["file_name"]

    present = img_to_objs[img_id]

    if not present:
        continue

    # ✅ generate 3 positive
    for _ in range(3):
        cat = random.choice(list(present))
        data.append({
            "image": filename,
            "question": f"Is there a {cat_map[cat]} in the image?",
            "label": "yes"
        })

    # ❌ generate 3 negative
    absent = list(set(all_categories) - present)

    for _ in range(3):
        cat = random.choice(absent)
        data.append({
            "image": filename,
            "question": f"Is there a {cat_map[cat]} in the image?",
            "label": "no"
        })

# save
with open("pope_val_full.json", "w") as f:
    json.dump(data, f)

print("Total samples:", len(data))

In [ ]:
yes = sum(1 for d in data if d["label"] == "yes")
no  = sum(1 for d in data if d["label"] == "no")

print("YES:", yes)
print("NO :", no)

In [ ]:
import json

with open("pope_val_full.json") as f:
    data = json.load(f)

print(len(data))

In [ ]:
from transformers import BlipProcessor, BlipForQuestionAnswering
from PIL import Image

# Load pretrained BLIP
processor = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
model = BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base")

# Put model in eval mode
model.eval()

In [ ]:
from PIL import Image
from tqdm import tqdm
import os

model = model.to(device)   

hallucinations = 0
total_no = 0

hallucinations_list = []   # ⭐ store cases

base_path = "coco/val2014/"

for sample in tqdm(data[:5000]):   # start small
    
    image_path = os.path.join(base_path, sample["image"])
    question = sample["question"]
    gt = sample["label"]

    image = Image.open(image_path).convert("RGB")

    inputs = processor(image, question, return_tensors="pt").to(device)

    with torch.no_grad():
        out = model.generate(**inputs)

    pred = processor.decode(out[0], skip_special_tokens=True).lower()

    # normalize
    if "yes" in pred:
        pred = "yes"
    else:
        pred = "no"

    # check hallucination
    if gt == "no":
        total_no += 1

        if pred == "yes":
            hallucinations += 1

            # ⭐ SAVE CASE
            hallucinations_list.append({
                "image": sample["image"],
                "question": question,
                "gt": gt,
                "pred": pred
            })
            
print("Hallucinations:", hallucinations)
print("Total NO:", total_no)

In [ ]:
print(hallucinations_list[1])

In [ ]:
# Load image
image = Image.open("coco/val2014/COCO_val2014_000000005802.jpg").convert("RGB")

# Question
question = "Is there a wine glass in the image?"

model = model.to("cpu")

# Preprocess
inputs = processor(image, question, return_tensors="pt")

# Inference
with torch.no_grad():
    out = model.generate(**inputs)

# Decode answer
answer = processor.decode(out[0], skip_special_tokens=True)

print("Answer:", answer)

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(image)
plt.axis('off') 
plt.show()

print(question)
print(answer)

In [ ]:
# ==============================
# ATTENTION VISUALIZATION CELL
# ==============================

import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# ---- Forward pass WITH attentions ----
with torch.no_grad():
    vision_outputs = model.vision_model(
        pixel_values=inputs["pixel_values"],
        output_attentions=True
    )


# ---- Get vision encoder attentions ----
attentions = vision_outputs.attentions  # tuple

# ---- Take last layer ----
attn = attentions[-1]   # (B, heads, 197, 197)

# ---- Remove batch ----
attn = attn[0]          # (heads, 197, 197)

# ---- CLS → patches ----
cls_attn = attn[:, 0, 1:]   # (heads, 196)

# ---- Average heads ----
cls_attn = cls_attn.mean(dim=0)  # (196,)

# ---- Normalize ----
cls_attn = cls_attn / cls_attn.max()

# ---- Convert to grid ----
attn_map = cls_attn.reshape(24, 24).cpu().numpy()

# ---- Upsample ----
attn_map = torch.tensor(attn_map).unsqueeze(0).unsqueeze(0)
attn_map = F.interpolate(attn_map, size=image.size[::-1], mode='bilinear', align_corners=False)
attn_map = attn_map.squeeze().numpy()

# ---- Overlay ----
plt.figure(figsize=(6,6))
plt.imshow(image)
plt.imshow(attn_map, cmap='jet', alpha=0.5)
plt.axis('off')
plt.title("Attention Map")
plt.show()

In [ ]:
# ==============================
# CLIP SIMILARITY CELL
# ==============================

from transformers import CLIPProcessor, CLIPModel
import torch

# Load CLIP
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

clip_model.eval()

# ---- Prepare inputs ----
text = f"a photo of {answer}"   # VERY IMPORTANT
# text = "a photo of stone"

clip_inputs = clip_processor(
    text=[text],
    images=image,
    return_tensors="pt",
    padding=True
)

# ---- Forward pass ----
with torch.no_grad():
    outputs = clip_model(**clip_inputs)

# ---- Get embeddings ----
image_embeds = outputs.image_embeds
text_embeds = outputs.text_embeds

# ---- Normalize ----
image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

# ---- Cosine similarity ----
similarity = (image_embeds * text_embeds).sum().item()

print("Answer:", answer)
print("Text:", text)
print("CLIP similarity:", similarity)

In [ ]:
# ==============================
# CLEAN GRAD-CAM (NO ERRORS)
# ==============================

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

model.eval()

# ---- Forward pass (NO torch.no_grad) ----
vision_outputs = model.vision_model(
    pixel_values=inputs["pixel_values"],
    output_hidden_states=True,
    return_dict=True
)

# ---- Get features ----
features = vision_outputs.last_hidden_state  # (1, 577, dim)

# Remove CLS
patch_features = features[:, 1:, :]  # (1, 576, dim)

# ---- Create pseudo score ----
score = patch_features.mean()

# ---- Get gradients (SAFE way) ----
grads = torch.autograd.grad(
    outputs=score,
    inputs=patch_features,
    retain_graph=False,
    create_graph=False
)[0]  # (1, 576, dim)

# ---- Compute weights ----
weights = grads.mean(dim=-1)  # (1, 576)

# ---- Compute CAM ----
cam = weights * patch_features.mean(dim=-1)  # (1, 576)
cam = cam.squeeze()

# ---- ReLU ----
cam = torch.relu(cam)

# ---- Normalize ----
cam = cam / cam.max()

# ---- Reshape ----
grid_size = int(cam.shape[0] ** 0.5)
cam = cam.reshape(grid_size, grid_size).detach().cpu().numpy()

# ---- Upsample ----
cam = torch.tensor(cam).unsqueeze(0).unsqueeze(0)
cam = F.interpolate(cam, size=image.size[::-1], mode='bilinear', align_corners=False)
cam = cam.squeeze().numpy()

# ---- Overlay ----
plt.figure(figsize=(6,6))
plt.imshow(image)
plt.imshow(cam, cmap='jet', alpha=0.5)
plt.axis('off')
plt.title("Grad-CAM")
plt.show()



In [ ]:
# ---------------------------
# Contrastive Decoding Cell
# ---------------------------

alpha = 0.5  # tune this (0.3–0.7)

# 1. Generate multiple candidate answers
inputs = processor(image, question, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        num_beams=5,
        num_return_sequences=5,
        max_length=20,
        early_stopping=True
    )

candidates = processor.batch_decode(outputs, skip_special_tokens=True)

print("Candidates:")
for c in candidates:
    print("-", c)


# 2. Create dummy image (text-only case)
dummy_image = Image.new("RGB", image.size, (0, 0, 0))


# 3. Function to compute score
def compute_score(img, question, answer):
    inputs = processor(img, question, return_tensors="pt")
    labels = processor.tokenizer(answer, return_tensors="pt").input_ids

    with torch.no_grad():
        outputs = model(**inputs, labels=labels)

    score = -outputs.loss.item()

    # length normalization
    score /= labels.shape[1]

    return score


# 4. Rerank using contrastive scoring
best_answer = None
best_score = -1e9

print("\nScoring:\n")

for ans in candidates:
    score_img = compute_score(image, question, ans)
    score_txt = compute_score(dummy_image, question, ans)

    final_score = score_img - alpha * score_txt

    print(f"Answer: {ans}")
    print(f"  img_score: {score_img:.4f}")
    print(f"  txt_score: {score_txt:.4f}")
    print(f"  final:     {final_score:.4f}")
    print("-" * 40)

    if final_score > best_score:
        best_score = final_score
        best_answer = ans


print("\nFinal Answer (Contrastive Decoding):")
print(best_answer)